# 365 Probabilidades - Dia #103
## Qual a probabilidade de você escolher a fila errada?

**Tipo:** Comportamental  
**Data de publicação:** 2026-09-24  
**Ferramenta:** Python  
**Decisão analisada:** Qual fila escolher no supermercado, e adianta escolher?  
**Hashtag:** #365Probabilidades #Dia103

---

### A História

Todo mundo tem essa memória.

O carrinho está cheio, três caixas abertos, e a escolha parece trivial. Olha-se
para a esquerda, olha-se para a direita, conta-se gente. Escolhe-se. E dois
minutos depois a pessoa que chegou junto, e foi para a fila ao lado, já está
guardando o troco.

A sensação universal é de azar pessoal. Como se existisse uma regra segundo a
qual a outra fila sempre anda mais.

Não existe regra nenhuma. Existe aritmética, e existe um detalhe sobre o que a
gente consegue ver na hora de escolher.

---

### O Conceito: o que você vê não é o que decide

Na hora de escolher uma fila, o que se observa é o **número de pessoas** em
cada uma. O que decide o tempo de espera é o **tamanho das compras** de quem está
na frente, que não dá para ver do lugar onde a decisão é tomada. Duas pessoas com
carrinho cheio andam mais devagar que quatro com cestinha.

Isso não é só intuição. Um estudo de campo num supermercado, com a fila medida
por reconhecimento de vídeo, mostrou que os clientes reagem principalmente ao
**comprimento** da fila e não ajustam o suficiente pela **velocidade** com que
ela anda.

É por isso que a simulação deste dia faz os clientes escolherem a fila com menos
gente: é o comportamento documentado, não uma suposição arbitrária.

---

### O Modelo

Este dia é uma **simulação**, e isso precisa estar claro antes de qualquer
número. A simulação não mede o mundo: ela define premissas e mostra o que
decorre delas. Os números são propriedades do modelo, não medições de
supermercados reais. O que vale é o mecanismo.

Premissas, todas declaradas: três caixas, chegadas em processo de Poisson,
tempo de atendimento exponencial com média de 2,5 minutos, sem desistência, sem
troca de fila no meio do caminho, os primeiros 10% dos clientes descartados como
aquecimento, semente 42 e mais nove sementes para medir o erro de Monte Carlo.

**Fonte:**
- Lu, Y., Musalem, A., Olivares, M. & Schilkrut, A. (2013). Measuring the Effect
  of Queues on Customer Purchases. *Management Science*, 59(8), 1743-1763.
  Seção de frios de um grande supermercado numa metrópole latino-americana; fila
  medida por reconhecimento de vídeo e cruzada com os registros de venda. Os
  clientes olham principalmente o comprimento da fila, sem ajustar o suficiente
  pela velocidade. **N não localizado no resumo publicado.** A fonte entra como
  evidência do mecanismo de escolha, não como origem de número.

**O resto é aritmética e simulação.** Com c filas simétricas, a chance de a sua
ser a mais rápida é 1/c, e a de alguma outra ser melhor é 1 − 1/c. A simulação é
conferida contra a fórmula fechada do sistema M/M/1.

**Fator de correção ×0,80:** não se aplica. Saída de simulação, sem autorrelato.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 42

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO = '#c0392b'
DOURADO = '#c8a84b'
VERDE = '#1a5f5a'
CINZA = '#6b6a64'

def mil(n):
    return f"{n:,}".replace(",", ".")

print("Bibliotecas carregadas. Semente fixa:", SEED)


In [ ]:
# --- PREMISSAS DO MODELO ---
# ATENCAO: tudo nesta celula e ESCOLHA, nao medicao. A simulacao define o
# mundo dela. A fonte do dia (Lu et al., 2013) sustenta o MECANISMO de
# escolha (olhar o comprimento da fila), nao nenhum numero abaixo.

C_CAIXAS = 3                 # caixas abertos no cenario principal
SERVICO_MEDIO = 2.5          # minutos por cliente
MU = 1 / SERVICO_MEDIO       # taxa de atendimento por caixa
N_CLIENTES = 40_000          # clientes no cenario principal
UTILIZACOES = [0.5, 0.6, 0.7, 0.8, 0.9]   # ocupacao dos caixas
AQUECIMENTO = 0.10           # descarta os primeiros 10% (a loja comeca vazia)
N_SEMENTES = 10              # replicacoes para o erro de Monte Carlo
N_CLIENTES_REPLICA = 20_000
CAIXAS_TESTADOS = [2, 3, 4, 5, 6]
CV_ALTERNATIVO = 0.5         # compras menos variaveis que a exponencial (CV = 1)

aplica_fator_080 = False

print("=" * 68)
print("  PREMISSAS DA SIMULACAO (escolhas, nao medicoes)")
print("=" * 68)
print(f"\n  Caixas abertos (cenario principal): {C_CAIXAS}")
print(f"  Atendimento medio:                  {SERVICO_MEDIO} min")
print(f"  Chegadas:                           processo de Poisson")
print(f"  Clientes por cenario:               {mil(N_CLIENTES)}")
print(f"  Aquecimento descartado:             {AQUECIMENTO*100:.0f}%")
print(f"  Sementes:                           {SEED} e mais {N_SEMENTES - 1}")
print(f"  Sem desistencia, sem troca de fila no meio do caminho.")
print(f"\n  Fonte do mecanismo: Lu et al., 2013, Management Science")
print(f"  -> clientes olham o comprimento da fila, nao a velocidade")
print(f"\n  Fator x0,80: {'SIM' if aplica_fator_080 else 'NAO'}")
print("=" * 68)


In [ ]:
# --- O MODELO ---

def simula_filas(n, c, lam, mu, politica, semente, cv=1.0):
    '''
    Cada cliente escolhe uma fila e fica nela ate ser atendido.
      politica 'curta'     -> fila com menos gente (o que a pessoa ve)
      politica 'aleatoria' -> escolhe sem olhar
    Para cada cliente calcula tambem o que teria acontecido nas OUTRAS filas
    com o mesmo tempo de compra (contrafactual). Aproximacao declarada: o
    contrafactual ignora o efeito da escolha sobre quem vem depois.
    'A fila ao lado' e a fila imediatamente a direita da escolhida.
    cv = coeficiente de variacao do atendimento (1 = exponencial).
    '''
    g = np.random.default_rng(semente)
    chegadas = np.cumsum(g.exponential(1 / lam, n))
    if cv == 1.0:
        servicos = g.exponential(1 / mu, n)
    else:
        forma = 1 / cv ** 2
        servicos = g.gamma(forma, (1 / mu) / forma, n)

    livre = np.zeros(c)
    pessoas = np.zeros(c, dtype=int)
    saida = np.empty(n)
    errou = np.empty(n, dtype=bool)
    ao_lado_melhor = np.empty(n, dtype=bool)
    perdido = np.empty(n)

    for i in range(n):
        t = chegadas[i]
        visivel = np.array([pessoas[k] if livre[k] > t else 0 for k in range(c)])
        if politica == 'curta':
            candidatos = np.flatnonzero(visivel == visivel.min())
        else:
            candidatos = np.arange(c)
        k = int(g.choice(candidatos))

        contrafactual = np.maximum(t, livre) + servicos[i]
        dep = contrafactual[k]
        outras = np.delete(contrafactual, k)

        saida[i] = dep
        errou[i] = outras.min() < dep - 1e-9
        ao_lado_melhor[i] = contrafactual[(k + 1) % c] < dep - 1e-9
        perdido[i] = dep - contrafactual.min()

        livre[k] = dep
        for kk in range(c):
            pessoas[kk] += (kk == k)
            if livre[kk] <= t:
                pessoas[kk] = 0

    espera = saida - chegadas - servicos
    corte = int(n * AQUECIMENTO)
    return dict(errou=errou[corte:], ao_lado=ao_lado_melhor[corte:],
                perdido=perdido[corte:], espera=espera[corte:])

def espera_mm1(c, lam, mu):
    'Espera media na fila de c filas M/M/1 independentes (escolha aleatoria).'
    lam_i = lam / c
    return lam_i / (mu * (mu - lam_i))

# ---------------------------------------------------------------
# PARTE A e B - cenario principal, 3 caixas
# ---------------------------------------------------------------
res = {}
for rho in UTILIZACOES:
    lam = rho * C_CAIXAS * MU
    for pol in ['aleatoria', 'curta']:
        r = simula_filas(N_CLIENTES, C_CAIXAS, lam, MU, pol, SEED)
        res[(rho, pol)] = dict(errou=r['errou'].mean(), ao_lado=r['ao_lado'].mean(),
                               perdido=r['perdido'], espera=r['espera'].mean())

# TESTE DO SIMULADOR: escolha aleatoria => cada fila e um M/M/1 exato
teste = {rho: (espera_mm1(C_CAIXAS, rho * C_CAIXAS * MU, MU), res[(rho, 'aleatoria')]['espera'])
         for rho in UTILIZACOES}

# ERRO DE MONTE CARLO
replicas = {}
for rho in [0.5, 0.8, 0.9]:
    lam = rho * C_CAIXAS * MU
    for pol in ['aleatoria', 'curta']:
        v = [simula_filas(N_CLIENTES_REPLICA, C_CAIXAS, lam, MU, pol, s)['errou'].mean()
             for s in range(SEED, SEED + N_SEMENTES)]
        replicas[(rho, pol)] = (np.mean(v), np.std(v, ddof=1), min(v), max(v))

# ---------------------------------------------------------------
# PARTE C - mais caixas, mais chance de errar (ocupacao de 80%)
# ---------------------------------------------------------------
por_caixas = {}
for c in CAIXAS_TESTADOS:
    lam = 0.8 * c * MU
    for pol in ['aleatoria', 'curta']:
        por_caixas[(c, pol)] = simula_filas(N_CLIENTES_REPLICA, c, lam, MU, pol, SEED)['errou'].mean()

# ---------------------------------------------------------------
# PARTE D - sensibilidade: compras menos variaveis (CV = 0,5)
# Testa se o resultado depende da premissa exponencial (CV = 1).
# ---------------------------------------------------------------
sens_cv = {}
for rho in [0.5, 0.8, 0.9]:
    lam = rho * C_CAIXAS * MU
    for pol in ['aleatoria', 'curta']:
        sens_cv[(rho, pol)] = simula_filas(N_CLIENTES_REPLICA, C_CAIXAS, lam, MU, pol, SEED,
                                           cv=CV_ALTERNATIVO)['errou'].mean()

# arrependimento no cenario de 80%, olhando a fila
perdido80 = res[(0.8, 'curta')]['perdido']
perdido80 = perdido80[perdido80 > 0.01]
arrep_mediana = np.median(perdido80)
arrep_p95 = np.percentile(perdido80, 95)

print("=" * 68)
print("  TESTE DO SIMULADOR (escolha aleatoria contra a formula de M/M/1)")
print("=" * 68)
print(f"\n  {'Ocupacao':<10}{'formula':>12}{'simulacao':>12}{'diferenca':>12}")
for rho, (f, s) in teste.items():
    print(f"  {rho*100:>5.0f}%    {f:>10.2f}{s:>12.2f}{(s/f-1)*100:>11.1f}%")

print("\n" + "=" * 68)
print("  PARTE A - A CHANCE DE ESCOLHER A FILA ERRADA (3 caixas)")
print("=" * 68)
print(f"\n  'Errou' = alguma outra fila teria devolvido o cliente mais cedo,")
print(f"  com o mesmo tamanho de compra.\n")
print(f"  {'Ocupacao':<10}{'sem olhar':>12}{'olhando a fila':>18}")
for rho in UTILIZACOES:
    print(f"  {rho*100:>5.0f}%    {res[(rho,'aleatoria')]['errou']*100:>10.1f}%"
          f"{res[(rho,'curta')]['errou']*100:>16.1f}%")
print(f"\n  Aritmetica pura, escolha cega: {100*(1-1/C_CAIXAS):.1f}%")
print(f"\n  Quanto olhar reduz o erro:")
for rho in UTILIZACOES:
    ganho = (res[(rho,'aleatoria')]['errou'] - res[(rho,'curta')]['errou']) * 100
    print(f"  -> {rho*100:>3.0f}% de ocupacao: {ganho:4.1f} pontos percentuais")
print(f"\n  Erro de Monte Carlo ({N_SEMENTES} sementes, {mil(N_CLIENTES_REPLICA)} clientes cada):")
for (rho, pol), (m, dp, lo, hi) in replicas.items():
    rot = 'olhando a fila' if pol == 'curta' else 'sem olhar'
    print(f"  -> {rho*100:>3.0f}%, {rot:<15} media {m*100:5.1f}%  dp {dp*100:.2f} p.p.  "
          f"faixa [{lo*100:.1f}; {hi*100:.1f}]")

print("\n" + "=" * 68)
print("  PARTE B - A FILA AO LADO CONTRA A MELHOR DAS OUTRAS")
print("=" * 68)
print(f"\n  {'Ocupacao':<10}{'politica':<16}{'a fila ao lado':>16}{'alguma das outras':>20}")
for rho in UTILIZACOES:
    for pol in ['aleatoria', 'curta']:
        rot = 'olhando' if pol == 'curta' else 'sem olhar'
        print(f"  {rho*100:>5.0f}%    {rot:<16}{res[(rho,pol)]['ao_lado']*100:>14.1f}%"
              f"{res[(rho,pol)]['errou']*100:>18.1f}%")
print(f"\n  Aritmetica pura: a fila ao lado 1/2, alguma das outras duas 2/3.")
print(f"  A sensacao de que 'a outra fila sempre anda mais' vem da segunda")
print(f"  conta: a gente compara com a melhor das outras, nao com uma so.")

print("\n" + "=" * 68)
print("  PARTE C - MAIS CAIXAS: OLHAR VALE MAIS (ocupacao de 80%)")
print("=" * 68)
print(f"\n  {'Caixas':<8}{'aritmetica':>12}{'sem olhar':>12}{'olhando a fila':>18}")
for c in CAIXAS_TESTADOS:
    print(f"  {c:<8}{100*(1-1/c):>10.1f}%{por_caixas[(c,'aleatoria')]*100:>11.1f}%"
          f"{por_caixas[(c,'curta')]*100:>16.1f}%")
print(f"\n  Sem olhar, cada caixa a mais aumenta o erro, como manda a aritmetica.")
print(f"  Olhando a fila, o erro CAI: com mais opcoes, a chance de haver uma")
print(f"  fila bem mais curta aumenta, e olhar passa a valer mais.")

print("\n" + "=" * 68)
print(f"  PARTE D - SENSIBILIDADE: COMPRAS MENOS VARIAVEIS (CV = {CV_ALTERNATIVO})")
print("=" * 68)
print(f"\n  {'Ocupacao':<10}{'politica':<12}{'CV = 1 (exponencial)':>22}{'CV = 0,5':>12}")
for rho in [0.5, 0.8, 0.9]:
    for pol in ['aleatoria', 'curta']:
        rot = 'olhando' if pol == 'curta' else 'sem olhar'
        print(f"  {rho*100:>5.0f}%    {rot:<12}{res[(rho,pol)]['errou']*100:>20.1f}%"
              f"{sens_cv[(rho,pol)]*100:>11.1f}%")
print(f"\n  O erro quase nao muda com compras menos variaveis. O resultado nao")
print(f"  depende da premissa exponencial; a 90% de ocupacao a diferenca fica")
print(f"  dentro do erro de Monte Carlo medido na parte A.")

print("\n" + "=" * 68)
print("  O TAMANHO DO ARREPENDIMENTO (80% de ocupacao, olhando a fila)")
print("=" * 68)
print(f"\n  Mediana de quem errou:        {arrep_mediana:.1f} min")
print(f"  1 em cada 20 perde mais de:   {arrep_p95:.1f} min")
print("=" * 68)


In [ ]:
# --- VISUALIZACAO ---
rhos = np.array(UTILIZACOES) * 100

# GRAFICO 1 - A chance de escolher a fila errada
fig1, ax1 = plt.subplots(figsize=(10, 6))
e_ale = [res[(r, 'aleatoria')]['errou'] * 100 for r in UTILIZACOES]
e_cur = [res[(r, 'curta')]['errou'] * 100 for r in UTILIZACOES]
ax1.plot(rhos, e_ale, color=VERMELHO, linewidth=2.8, marker='o', markersize=8,
         label='escolhendo sem olhar')
ax1.plot(rhos, e_cur, color=VERDE, linewidth=2.8, marker='o', markersize=8,
         label='escolhendo a fila com menos gente')
ax1.axhline(y=100 * (1 - 1 / C_CAIXAS), color=DOURADO, linestyle='--', linewidth=2)
ax1.text(51, 100 * (1 - 1 / C_CAIXAS) + 2, 'a aritmetica pura: 2 em 3 com 3 caixas',
         fontsize=10, color=DOURADO)
ax1.set_xlabel('Ocupacao dos caixas (%)')
ax1.set_ylabel('Chance de ter escolhido a fila errada (%)')
ax1.set_ylim(8, 76)
ax1.legend(loc='lower right', fontsize=10, frameon=False)
ax1.set_title('Olhar a fila ajuda, ate a loja encher\nSimulacao: 3 caixas, 40.000 clientes por cenario, semente 42',
              fontsize=12, pad=15)
plt.figtext(0.5, 0.005, 'Saida de simulacao, nao medicao de supermercados reais | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-103-grafico-01-fila-errada.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo.")

# GRAFICO 2 - A fila ao lado contra a melhor das outras
fig2, ax2 = plt.subplots(figsize=(10, 6))
for pol, cor, rot in [('aleatoria', VERMELHO, 'sem olhar'), ('curta', VERDE, 'olhando a fila')]:
    ax2.plot(rhos, [res[(r, pol)]['errou'] * 100 for r in UTILIZACOES], color=cor,
             linewidth=2.8, marker='o', label=f'alguma das outras foi melhor ({rot})')
    ax2.plot(rhos, [res[(r, pol)]['ao_lado'] * 100 for r in UTILIZACOES], color=cor,
             linewidth=2.2, linestyle='--', marker='s', label=f'a fila ao lado foi melhor ({rot})')
ax2.axhline(y=200 / 3, color=DOURADO, linestyle=':', linewidth=1.8)
ax2.axhline(y=50, color=DOURADO, linestyle=':', linewidth=1.8)
ax2.text(92.3, 200 / 3, '2/3', va='center', fontsize=10, color=DOURADO)
ax2.text(92.3, 50, '1/2', va='center', fontsize=10, color=DOURADO)
ax2.set_xlim(48, 95)
ax2.set_ylim(0, 76)
ax2.set_xlabel('Ocupacao dos caixas (%)')
ax2.set_ylabel('Chance (%)')
ax2.legend(loc='lower right', fontsize=9, frameon=False)
ax2.set_title('A fila ao lado ganha metade das vezes; alguma das outras, duas em tres\nA sensacao de "sempre" vem de comparar com a melhor das outras',
              fontsize=12, pad=15)
plt.figtext(0.5, 0.005, 'Saida de simulacao, nao medicao de supermercados reais | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-103-grafico-02-ao-lado-contra-a-melhor.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo.")

# GRAFICO 3 - Mais caixas, mais chance de errar
fig3, ax3 = plt.subplots(figsize=(10, 6))
cs = np.array(CAIXAS_TESTADOS)
ax3.plot(cs, 100 * (1 - 1 / cs), color=DOURADO, linewidth=2.2, linestyle='--', marker='D',
         label='aritmetica pura: 1 - 1/caixas')
ax3.plot(cs, [por_caixas[(c, 'aleatoria')] * 100 for c in CAIXAS_TESTADOS], color=VERMELHO,
         linewidth=2.8, marker='o', label='simulacao, escolhendo sem olhar')
ax3.plot(cs, [por_caixas[(c, 'curta')] * 100 for c in CAIXAS_TESTADOS], color=VERDE,
         linewidth=2.8, marker='o', label='simulacao, olhando a fila')
for c in CAIXAS_TESTADOS:
    ax3.text(c, por_caixas[(c, 'curta')] * 100 - 4.5, f"{por_caixas[(c, 'curta')]*100:.0f}%",
             ha='center', fontsize=10, color=VERDE)
ax3.set_xticks(cs)
ax3.set_ylim(0, 100)
ax3.set_xlabel('Caixas abertos')
ax3.set_ylabel('Chance de ter escolhido a fila errada (%)')
ax3.legend(loc='lower right', fontsize=10, frameon=False)
ax3.set_title('Com mais caixas, quem escolhe sem olhar erra mais, e quem olha erra menos\nOcupacao de 80%, 20.000 clientes por cenario, semente 42',
              fontsize=12, pad=15)
plt.figtext(0.5, 0.005, 'Saida de simulacao, nao medicao de supermercados reais | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-103-grafico-03-mais-caixas.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo.")

# GRAFICO 4 - O tamanho do arrependimento
fig4, ax4 = plt.subplots(figsize=(10, 6))
ax4.hist(perdido80, bins=70, color=VERDE, alpha=0.6, edgecolor='white', linewidth=0.3)
topo = ax4.get_ylim()[1]
ax4.axvline(x=arrep_mediana, color=VERMELHO, linewidth=2.5)
ax4.text(arrep_mediana + 0.5, topo * 0.85,
         f'mediana de quem errou:\n{arrep_mediana:.1f} min'.replace('.', ','),
         fontsize=11, color=VERMELHO)
ax4.axvline(x=arrep_p95, color=DOURADO, linestyle='--', linewidth=2)
ax4.text(arrep_p95 + 0.5, topo * 0.45,
         f'1 em 20 perde mais de\n{arrep_p95:.0f} min', fontsize=11, color=DOURADO)
ax4.set_xlim(0, np.percentile(perdido80, 99.5))
ax4.set_xlabel('Minutos perdidos em relacao a melhor fila disponivel')
ax4.set_ylabel('Clientes na simulacao')
ax4.set_title('O tamanho do arrependimento\nOcupacao de 80%, escolhendo a fila com menos gente',
              fontsize=12, pad=15)
plt.figtext(0.5, 0.005, 'Saida de simulacao, nao medicao de supermercados reais | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-103-grafico-04-arrependimento.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 4 salvo.")


### O Insight

**Duas em cada três vezes, se você escolher sem olhar.**

Com três caixas abertos, só uma fila pode ser a mais rápida. Quem escolhe no
escuro erra 2 em cada 3 vezes, e a simulação confirma a aritmética: 63% com o
mercado em 80% de ocupação.

**Olhar a fila ajuda, e ajuda cada vez menos conforme a loja enche.** Com
movimento normal, quem escolhe a fila com menos gente erra 44% das vezes: olhar
reduz o erro em 20 pontos. Com a loja cheia, o erro volta a 60%, e olhar reduz
só 6 pontos. O que a gente vê é o número de pessoas; o que decide é o tamanho
das compras delas, que ninguém enxerga.

E a sensação de que a outra fila **sempre** anda mais tem uma explicação
simples. A fila ao lado ganha de você mais ou menos metade das vezes. Mas a
gente não compara com a fila ao lado, compara com a melhor das outras, e alguma
delas vence duas em cada três.

Um detalhe inesperado: com mais caixas abertos, quem escolhe sem olhar erra
cada vez mais (76% com seis caixas). Quem olha, a partir de três caixas, erra
cada vez menos (36% com seis). Olhar vale mais quanto mais opções existem.

Quando o erro acontece, custa pouco na maioria das vezes: 6 minutos na mediana.
Mas 1 em cada 20 perde mais de 28.

A pergunta que fica:

*Quantas escolhas você refaz na cabeça achando que decidiu mal, quando o que
faltou foi a informação que ninguém tinha?*

---

### Limitações do Modelo

- **Simulação define, não mede.** Chegadas de Poisson, atendimento de 2,5
  minutos em média e três caixas são premissas escritas no código. Os números
  são propriedades do modelo; o que sobrevive fora dele é o mecanismo.
- **A premissa exponencial não mexe no resultado.** Compras reais variam menos
  que uma exponencial. Refeita com variação menor (CV = 0,5, parte D), a
  simulação dá praticamente os mesmos números; a 90% de ocupação a diferença
  fica dentro do erro de Monte Carlo. Antes do teste, a hipótese era que compras
  menos variáveis favoreceriam quem olha a fila. Não favoreceram.
- **O contrafactual é aproximado.** Para saber se o cliente escolheu bem, o
  modelo calcula o que teria acontecido nas outras filas mantendo o resto igual,
  o que ignora o efeito da escolha sobre quem vem depois.
- **"A fila ao lado" é a imediatamente à direita.** Numa loja real, "a fila ao
  lado" é a que a pessoa está olhando, e a atenção tende a ir para a que andou,
  o que só reforça a diferença entre as duas contas.
- **Sem troca de fila.** Na vida real, trocar de fila no meio do caminho reduz
  parte do prejuízo de ter escolhido mal.
- **Validação do simulador.** Com escolha aleatória, cada fila é um M/M/1
  exato, e a espera simulada bate com a fórmula fechada (teste no output). Os
  primeiros 10% dos clientes são descartados como aquecimento, e a variação
  entre dez sementes é reportada.
- **A fonte sustenta o mecanismo, não os números.** Lu et al. (2013) mediram,
  no balcão de frios de um supermercado, a reação dos clientes ao tamanho da
  fila na decisão de compra, não a escolha entre caixas. O N não aparece no
  resumo publicado, e o artigo completo não foi auditado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
